# Import Data

In [1]:
import sklearn as sk
import pandas as pd
import numpy as np
df = pd.read_parquet("basic_processed_data.parquet")

# Regression Target

In [2]:
df['Severity'] = df['Severity'].astype(float) * 10
y = df['Severity']
X = df.copy().drop(columns=['Severity'])

# Part III.1: Perform 10-Fold Cross Validation 

## Advanced Preprocessing

In [3]:
def add_road_type(f):
    HWY_RE = (
        r'\b[A-Z]{1,2}-\d+'                      # I-95, US-101, CA-99, WI-50
        r'|\b(?:Hwy|Fwy|Expy|Pkwy|Tpke|Turnpike|Highway|Freeway|Expressway|Interstate|Route|Rte|SR)\b'
    )
    
    LOCAL_RE = (
        r'\b(?:St|Street|Rd|Road|Ave|Avenue|Blvd|Boulevard|Ln|Lane|Dr|Drive|'
        r'Ct|Court|Pl|Place|Way|Pike|Cir|Circle|Trl|Trail|Ter|Terrace)\b'
    )   

    
    s = f['Street']
    f['road_type'] = np.select(
        [s.isna(), s.str.contains(HWY_RE, regex=True, na=False), s.str.contains(LOCAL_RE, regex=True, na=False)],
        [None, 'highway', 'local_street'],
        default='other'
    )
    
    return f

def weather_type(cond):
    if pd.isna(cond): return np.nan # or should i do "unknown"?
    elif ("Fair" in cond or cond == "Clear"): return "clear"
    elif ("Thunder" in cond or "T-Storm" in cond): return "storm"
    elif ("Rain" in cond or "Drizzle" in cond or "Precipitation" in cond or "Showers" in cond): return "rain"
    elif (cond == "Wintry Mix" or "Snow" in cond): return "snow"
    elif ("Cloudy" in cond or cond == "Cloudy" or "Clouds" in cond or "Overcast" in cond): 
        return "cloudy"
    elif ("Fog" in cond or "Mist" in cond or "Smoke" in cond or "Haze" in cond or "Dust" in cond or "Sand" in cond): 
        return "low-visibility"
    elif ("Sleet" in cond or "Hail" in cond or "Ice Pellets" in cond or "Tornado" in cond): 
        return "storm"
    elif ("Windy" in cond or "Squall" in cond): return "windy"
    else: return "cloudy"

def add_rush_hour(hour):
    if ((hour >= 7 and hour <= 9) or (hour >= 16 and hour <= 19)):
        return 1
    else: return 0

def advanced_preprocessing(fold):
    # Combine POI attributes into categories
    poi_type_groups = {
        "traffic_controlled": ['Traffic_Signal', 'Stop', 'Give_Way'],
        "junctions": ['Junction', 'Roundabout', 'No_Exit'],
        "traffic_calming": ['Bump', 'Traffic_Calming'],
        "pedestrians": ['Crossing', 'Station', 'Amenity']
    }
    
    F = fold.copy()
    for name, cols in poi_type_groups.items():
        F[name] = F[cols].fillna(False).astype(bool).any(axis=1).astype(int)
    
    F = F.drop(columns=poi_type_groups.get("traffic_controlled"))
    F = F.drop(columns=poi_type_groups.get("junctions"))
    F = F.drop(columns=poi_type_groups.get("traffic_calming"))
    F = F.drop(columns=poi_type_groups.get("pedestrians"))
    
    
    # Create road_type attribute from Street attribute
    F = add_road_type(F)
    F = F.drop(columns=['Street'])
    
    
    # Create time of day/lighting attribute from Sunrise_Sunset and Civil_Twilight
    F['tod_lighting'] = np.select(
        [F['Sunrise_Sunset'] == "Day", F['Civil_Twilight'] == "Day"],
        ['day', 'twilight'],
        default='night'
    )
    F = F.drop(columns=["Sunrise_Sunset", "Civil_Twilight","Nautical_Twilight", "Astronomical_Twilight"])
    
    
    # Create weather condition groups
    F['Weather_Condition'] = F['Weather_Condition'].apply(weather_type)
    
    
    # Create rush-hour flag from Start_Hour
    F['rush_hour'] = F['Start_Hour'].apply(add_rush_hour)
    
    return F




def listVals(f):
    vc = f["tod_lighting"].value_counts(dropna=False)
    print(len(vc), "unique values")
    print(vc.to_string())  






# Regression Techniques

In [19]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import FunctionTransformer, Pipeline
from sklearn.preprocessing import OneHotEncoder


def perform_cv(Model, X, y):
    kf = KFold(n_splits=10, shuffle=True, random_state=1) 

    scoring_metrics = ["neg_mean_squared_error", "neg_root_mean_squared_error", "neg_mean_absolute_error", "r2"]

    cat_cols = make_column_selector(dtype_include=object)
    num_cols = make_column_selector(dtype_exclude=object)

    # don't impute for anything but linear regression???
    encode = ColumnTransformer([
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))]), 
        cat_cols),
        ("num", SimpleImputer(strategy="median"), num_cols),
    ])

    pipe = Pipeline([
        ("prep", FunctionTransformer(advanced_preprocessing)),
        ("encode", encode),
        ("model", Model),
    ])

    res = cross_validate(pipe, X, y, cv=kf, scoring=scoring_metrics)
    return res


## Dummy Regressor

In [8]:
from sklearn.dummy import DummyRegressor

dr = DummyRegressor(strategy='mean')
res = perform_cv(dr)

print(res)

{'fit_time': array([18.70665622, 18.20439577, 18.21639919, 18.16375232, 18.27612877,
       18.18863654, 18.02643847, 18.49584103, 18.15817046, 18.1132648 ]), 'score_time': array([0.91857886, 0.90406108, 0.89991331, 0.91638422, 0.94801164,
       0.91056561, 0.91400313, 0.92097282, 0.89953876, 0.90232444]), 'test_neg_mean_squared_error': array([-16.42748148, -16.46092016, -16.33232274, -16.42375592,
       -16.59991025, -16.46095818, -16.66497206, -16.44312543,
       -16.52697062, -16.61196717]), 'test_neg_root_mean_squared_error': array([-4.05308296, -4.05720595, -4.04132685, -4.05262334, -4.07429874,
       -4.05721064, -4.08227535, -4.05501238, -4.0653377 , -4.07577811]), 'test_neg_mean_absolute_error': array([-1.78610723, -1.79067446, -1.78441696, -1.78845576, -1.79558281,
       -1.78995742, -1.79995942, -1.78877973, -1.79294304, -1.7965799 ]), 'test_r2': array([-1.56689319e-06, -8.90189051e-08, -4.91616118e-06, -2.86108504e-06,
       -2.39523027e-06, -2.00405319e-06, -1.1174426

## Linear Regression

In [9]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
res = perform_cv(lr)

print(res)

{'fit_time': array([31.07399464, 25.77412581, 23.25912213, 29.10241461, 23.63515377,
       30.45442176, 22.5192678 , 26.93913984, 31.50438285, 30.88487053]), 'score_time': array([1.27541184, 0.89503479, 1.24123478, 0.91348004, 1.25008082,
       1.24689484, 0.91257071, 1.22062564, 1.22955656, 1.23602009]), 'test_neg_mean_squared_error': array([-15.84234686, -15.88861152, -15.75750692, -15.85438701,
       -16.01192358, -15.87901194, -16.0632067 , -15.86460473,
       -15.93816428, -16.00072227]), 'test_neg_root_mean_squared_error': array([-3.98024457, -3.98605212, -3.96957264, -3.98175677, -4.00149017,
       -3.98484779, -4.00789305, -3.98303963, -3.99226305, -4.00009028]), 'test_neg_mean_absolute_error': array([-1.73303187, -1.73543796, -1.72908145, -1.73481643, -1.74193812,
       -1.73583579, -1.74638621, -1.73460626, -1.7380518 , -1.74207889]), 'test_r2': array([0.03561774, 0.03476763, 0.03519024, 0.03466464, 0.03541877,
       0.03535119, 0.03609882, 0.0351791 , 0.03562209, 0.03

## Regression Trees


In [21]:
from sklearn.tree import DecisionTreeRegressor

dtr1 = DecisionTreeRegressor(random_state=False)
dtr2 = DecisionTreeRegressor(max_depth=5, random_state=False)

# res1 = perform_cv(dtr1)
# res2 = perform_cv(dtr2)


Xs = X.sample(200_000, random_state=1)
ys = y.loc[Xs.index]
res1 = perform_cv(dtr1, Xs, ys)
res2 = perform_cv(dtr2, Xs, ys)

print(res1)
print('\n')
print(res2)

{'fit_time': array([3.43487978, 3.49426556, 3.46174335, 3.39994264, 3.41294622,
       3.35026932, 3.83037591, 3.69007015, 3.62155509, 3.53039765]), 'score_time': array([0.06247401, 0.05949545, 0.0600884 , 0.05839682, 0.05856228,
       0.05715847, 0.06189942, 0.05875707, 0.05960655, 0.06520033]), 'test_neg_mean_squared_error': array([-24.3       , -24.205     , -24.385     , -25.265     ,
       -24.16222222, -25.96      , -24.695     , -25.50625   ,
       -24.84      , -24.66      ]), 'test_neg_root_mean_squared_error': array([-4.92950302, -4.91985772, -4.93811705, -5.02643014, -4.91550834,
       -5.09509568, -4.9694064 , -5.05037127, -4.98397432, -4.96588361]), 'test_neg_mean_absolute_error': array([-1.381     , -1.3835    , -1.3765    , -1.4145    , -1.36133333,
       -1.466     , -1.4005    , -1.43875   , -1.397     , -1.381     ]), 'test_r2': array([-0.54898779, -0.45158555, -0.51735919, -0.51905985, -0.55580527,
       -0.5549037 , -0.57633573, -0.52584077, -0.49662502, -0.49

## Random Forests